# Setup check -- please run this before the Workshop

This short notebook checks that your Google Colab session can run the ASI-FIMSA spatial-omics Tutorials. It installs the software we use, downloads one tiny test file, and draws one small figure. It takes about two minutes and it changes nothing on your own computer.

**What to do:** click *Runtime -> Run all* in the menu, then wait. The last cell tells you either that you are ready, or exactly what to send us.

**You do not need a graphics card (GPU).** Everything in the Workshop runs on an ordinary free Colab session. One Tutorial trains a small neural network, and it is a few minutes faster with a GPU, but it is designed to finish comfortably without one.

**If something goes wrong, nothing is broken.** Colab sessions are disposable. You can always close the tab, open the notebook again and start over.

In [ ]:
# ============================================================================
# Step 1 of 4  --  install the software the Workshop uses
# ----------------------------------------------------------------------------
# This usually takes under a minute. Colab wipes everything you install when a
# session ends, so you will run a cell like this again at the start of each
# Tutorial on the day. That is expected, not a mistake.
#
# You do not need to read or understand anything below. Just run the cell.
# ============================================================================
import importlib
import pathlib
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules

# Exact versions we have tested together. Pinning them means everyone in the
# room is running the same software, so a problem on your screen is a problem
# we can reproduce on ours.
# --- BEGIN PINS (generated from constraints-colab.txt) ---
PINS = """\
numpy==2.0.2
spatialdata==0.8.0
spatialdata-io==0.7.1
spatialdata-plot==0.4.1
squidpy==1.8.3
sopa==2.2.10
scanpy==1.12.3
anndata==0.12.19
fast-array-utils==1.5
legacy-api-wrap==1.5
session-info2==0.4.2
array-api-compat==1.15.0
natsort==8.4.0
zarr==3.3.0
numcodecs==0.16.5
pydantic-zarr==0.10.0
ome-zarr==0.18.0
ome-zarr-models==1.7
ome-types==0.6.3
xsdata==26.2
xmltodict==1.0.4
xarray==2026.7.0
xarray-dataclass==3.0.0
xarray-spatial==0.10.17
spatial-image==1.2.3
multiscale-spatial-image==2.0.3
datashader==0.19.1
dask==2026.8.0
distributed==2026.8.0
dask-image==2026.5.0
fsspec==2026.7.0
universal-pathlib==0.3.10
toolz==1.1.0
partd==1.4.2
locket==1.0.0
cloudpickle==3.1.2
msgpack==1.2.1
zict==3.0.0
tblib==3.2.2
sortedcontainers==2.4.0
geopandas==1.1.4
shapely==2.1.2
pyogrio==0.13.0
pyproj==3.7.2
imagecodecs==2025.11.11
tifffile==2026.4.11
pims==0.7
slicerator==1.1.0
igraph==1.0.0
pynndescent==0.6.0
umap-learn==0.5.12
omnipath==1.0.12
validators==0.35.0
docrep==0.3.2
matplotlib-scalebar==0.9.0
typer==0.27.1
typeguard==4.6.0
pydantic==2.12.5
pydantic-core==2.41.5
bokeh==3.10.0
xyzservices==2026.3.0
narwhals==2.25.0
tornado==6.5.8
leidenalg==0.12.0
texttable==1.7.0
netgraph==4.13.2
rectangle-packer==2.1.0
grandalf==0.8
huggingface-hub==1.28.0
hf-xet==1.6.0
gdown==6.1.0
scipy>=1.13            # tested 1.18.1   <-- scanpy>=1.13
pandas>=2.3            # tested 2.3.3    <-- scanpy>=2.3  (forces a Colab upgrade)
matplotlib>=3.10       # tested 3.11.1   <-- scanpy>=3.10
scikit-learn>=1.6      # tested 1.9.0    <-- scanpy>=1.6
scikit-image>=0.25     # tested 0.26.0   <-- squidpy>=0.25
pillow>=8              # tested 12.3.0   <-- squidpy>=8
networkx>=2.8.8        # tested 3.6.1    <-- scanpy>=2.8.8
seaborn>=0.13.2        # tested 0.13.2   <-- scanpy>=0.13.2
numba>=0.60            # tested 0.67.0   <-- scanpy>=0.60
h5py>=3.11             # tested 3.16.0   <-- scanpy>=3.11
statsmodels>=0.13      # tested 0.14.6   <-- scanpy
joblib>=1.2            # tested 1.5.3    <-- scanpy
tqdm>=4.50.2           # tested 4.70.0   <-- squidpy>=4.50.2
requests>=2.28         # tested 2.34.2   <-- datashader, ome-zarr
pyarrow>=14            # tested 25.0.1   <-- spatialdata (parquet cell tables)
imageio>=2.37          # tested 2.37.4   <-- scikit-image, stlearn
click>=8.2             # tested 8.4.2    <-- typer (sopa), stlearn
"""
# --- END PINS ---

PACKAGES = [
    "spatialdata",
    "spatialdata-io",
    "spatialdata-plot",
    "squidpy",
    "sopa",
    "scanpy",
    "netgraph",
    "huggingface_hub",
    "gdown",
    "tifffile",
    "imagecodecs",
    "seaborn",
    "bokeh",
    "leidenalg",
]

# Packages Colab has already loaded into memory before you ran anything. If the
# install replaces one of these on disk, Python keeps using the old copy and
# things break in confusing ways later. We check for that at the end.
WATCH = ["numpy", "pandas", "scipy", "matplotlib", "sklearn", "PIL", "numba"]
DIST = {"sklearn": "scikit-learn", "PIL": "pillow"}


def _version(mod):
    from importlib.metadata import PackageNotFoundError, version
    try:
        return version(DIST.get(mod, mod))
    except PackageNotFoundError:
        return None


def _run(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-1500:])
        print(result.stderr[-1500:])
    return result.returncode


started = time.time()
preloaded = {m: _version(m) for m in WATCH if m in sys.modules}
INSTALL_OK = True

if IN_COLAB:
    constraints = pathlib.Path("/content/constraints-colab.txt")
    constraints.write_text(PINS)

    # uv is a much faster drop-in for pip. Installing uv itself costs a couple
    # of seconds and saves roughly half a minute, which matters when forty
    # people install at once on conference wifi. If anything about it goes
    # wrong we fall straight back to pip.
    rc = _run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    if rc == 0:
        rc = _run([sys.executable, "-m", "uv", "pip", "install", "--system", "-q",
                   "-c", str(constraints), *PACKAGES])
    if rc != 0:
        print("Falling back to pip (this is slower but works just as well)...")
        rc = _run([sys.executable, "-m", "pip", "install", "-q",
                   "-c", str(constraints), *PACKAGES])
    INSTALL_OK = rc == 0

    # stlearn, alone, with --no-deps. It declares numpy>=2.4.0, and Colab has
    # already imported numpy 2.0.2 -- satisfying that floor would force a session
    # restart. Everything stlearn actually imports is already installed above,
    # plus torchvision, which Colab preinstalls. ADR-0004 has the full argument.
    rc_stlearn = _run([sys.executable, "-m", "pip", "install", "-q",
                       "--no-deps", "stlearn"])
    if rc_stlearn != 0:
        print("stlearn did not install -- Tutorial 02b will not run.")
    INSTALL_OK = INSTALL_OK and rc_stlearn == 0
    importlib.invalidate_caches()
else:
    print("Not running in Google Colab, so nothing was installed.")
    print("This is the expected path when we test the notebook ourselves.")

# Did the install replace a package Python had already loaded?
STALE = [m for m, before in preloaded.items()
         if before is not None and _version(m) != before]

print(f"\nStep 1 finished in {time.time() - started:.0f} seconds.")
if not INSTALL_OK:
    print("The install reported an error. Keep going -- the checks below will")
    print("tell us exactly what is missing, and that is what we need from you.")
if STALE:
    print()
    print("=" * 70)
    print("PLEASE RESTART THE SESSION, THEN RUN THIS NOTEBOOK AGAIN FROM THE TOP.")
    print("Use the menu: Runtime -> Restart session.")
    print("Reason: these were updated while Python was already using them:")
    print("  " + ", ".join(STALE))
    print("Restarting takes a few seconds and is completely normal.")
    print("=" * 70)


## Step 2 -- check the software loaded correctly

The next cell imports each package and prints its version, then reports your Python version and whether this session happens to have a GPU. A row marked `MISSING` is the useful kind of failure: it tells us precisely what did not install.

In [ ]:
# ============================================================================
# Step 2 of 4  --  what is installed, what Python we have, and is there a GPU
# ============================================================================
import platform
import sys
from importlib.metadata import PackageNotFoundError, version

# import name -> the name it is installed under, where they differ
CHECKS = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("scipy", "scipy"),
    ("matplotlib", "matplotlib"), ("skimage", "scikit-image"),
    ("anndata", "anndata"), ("scanpy", "scanpy"), ("squidpy", "squidpy"),
    ("spatialdata", "spatialdata"), ("spatialdata_io", "spatialdata-io"),
    ("spatialdata_plot", "spatialdata-plot"), ("sopa", "sopa"),
    ("netgraph", "netgraph"), ("zarr", "zarr"), ("dask", "dask"),
    ("tifffile", "tifffile"), ("imagecodecs", "imagecodecs"),
    ("seaborn", "seaborn"), ("huggingface_hub", "huggingface_hub"),
    ("gdown", "gdown"), ("torch", "torch"), ("torchvision", "torchvision"),
    ("bokeh", "bokeh"), ("leidenalg", "leidenalg"), ("stlearn", "stlearn"),
]

print(f"Python {platform.python_version()}")
print()
print("{:<22}{:<16}{}".format("package", "version", "status"))
print("-" * 52)

MISSING = []
for import_name, dist_name in CHECKS:
    try:
        __import__(import_name)
        try:
            found = version(dist_name)
        except PackageNotFoundError:
            found = "installed"
        status = "ok"
    except Exception as exc:                      # noqa: BLE001
        found, status = "-", f"MISSING ({type(exc).__name__})"
        MISSING.append(dist_name)
    print(f"{dist_name:<22}{found:<16}{status}")

# numpy is the one version we care about exactly. Colab ships 2.0.2 and loads it
# before your first cell runs, so anything that quietly upgrades it forces a
# session restart in the middle of a Tutorial. If this line disagrees, tell us.
import numpy
NUMPY_OK = numpy.__version__ == "2.0.2"
print()
print(f"numpy is {numpy.__version__}"
      + ("" if NUMPY_OK else "  <-- we expected 2.0.2; please mention this to us"))

# A GPU is a bonus, never a requirement.
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else None
except Exception:                                 # noqa: BLE001
    HAS_GPU, GPU_NAME = False, None

print()
if HAS_GPU:
    print(f"GPU: yes ({GPU_NAME}). The neural-network Tutorial will run faster.")
else:
    print("GPU: no. That is completely fine -- everything is built to run without one.")
    print("If you would like one anyway: Runtime -> Change runtime type -> T4 GPU.")

IMPORTS_OK = not MISSING
print()
print("Step 2:", "all packages loaded" if IMPORTS_OK
      else f"{len(MISSING)} package(s) missing: {', '.join(MISSING)}")


## Step 3 -- check this session can reach the Workshop data

The Tutorials download prepared datasets while they run, so your Colab session needs to be able to reach the internet. This downloads one very small file to prove it can.

The Workshop dataset is still being uploaded, so it is completely normal for the first attempt below to say it cannot find it. In that case we fall back to a small public file, which tests the same thing.

In [ ]:
# ============================================================================
# Step 3 of 4  --  can this session download data?
# ============================================================================
from huggingface_hub import hf_hub_download

WORKSHOP_REPO = "xiao233333/asi-fimsa-workshop-2026"
DOWNLOAD_OK = False
downloaded_from = None

try:
    path = hf_hub_download(repo_id=WORKSHOP_REPO, filename="README.md",
                           repo_type="dataset")
    DOWNLOAD_OK, downloaded_from = True, "the Workshop dataset"
    print(f"Downloaded a file from {WORKSHOP_REPO}.")
except Exception as exc:                          # noqa: BLE001
    # Expected until the Workshop dataset is published. Deliberately no
    # traceback: a wall of red text reads like something you did wrong.
    print(f"Could not reach {WORKSHOP_REPO} yet ({type(exc).__name__}).")
    print("That is expected before the Workshop -- the dataset is still being")
    print("uploaded. Trying a small public file instead, which tests the same")
    print("thing: whether this session can download data at all.")
    try:
        path = hf_hub_download(repo_id="bert-base-uncased", filename="config.json")
        DOWNLOAD_OK, downloaded_from = True, "a public test file"
        print("That worked.")
    except Exception as exc2:                     # noqa: BLE001
        print(f"That did not work either ({type(exc2).__name__}: {exc2}).")
        print("This usually means a firewall or proxy is blocking the session.")
        print("Trying again on a different network often fixes it.")

if DOWNLOAD_OK:
    import os
    print(f"\nStep 3: downloads work ({downloaded_from}, "
          f"{os.path.getsize(path)} bytes).")
else:
    print("\nStep 3: downloads are blocked. Please tell us -- see the last cell.")


In [ ]:
# ============================================================================
# Step 3b  --  can this session draw a figure?
# ============================================================================
# Most of the Workshop is looking at pictures of tissue, so a session that
# cannot draw is a session that cannot follow along.
import matplotlib.pyplot as plt
import numpy as np

PLOT_OK = False
try:
    rng = np.random.default_rng(0)
    centres = np.array([[2.0, 2.0], [8.0, 3.0], [5.0, 8.0]])
    which = rng.integers(0, 3, 400)
    xy = centres[which] + rng.normal(0, 0.9, size=(400, 2))

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(xy[:, 0], xy[:, 1], c=which, cmap="viridis", s=14, edgecolor="none")
    ax.set_title("If you can see three blobs, plotting works")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect("equal")
    plt.show()
    PLOT_OK = True
except Exception as exc:                          # noqa: BLE001
    print(f"Plotting failed ({type(exc).__name__}: {exc})")

print("Step 3b:", "figure drawn" if PLOT_OK else "figure FAILED")


## Step 4 -- the verdict

Run the last cell. It collects the results of everything above into one answer.

In [ ]:
# ============================================================================
# Step 4 of 4  --  are you ready?
# ============================================================================
checks = {
    "software installed": IMPORTS_OK,
    "numpy is the expected version": NUMPY_OK,
    "downloads work": DOWNLOAD_OK,
    "figures draw": PLOT_OK,
}

print("=" * 70)
for label, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}   {label}")
print("=" * 70)
print()

if all(checks.values()):
    print("YOU ARE READY.")
    print()
    print("Nothing else to do before the Workshop. You do not need to keep this")
    print("session open -- each Tutorial installs what it needs when you open it.")
    if not HAS_GPU:
        print()
        print("You have no GPU, and that is fine. Everything is built to run without one.")
else:
    failed = [label for label, passed in checks.items() if not passed]
    print("NOT READY YET -- and this is usually quick for us to fix.")
    print()
    print("Please email us BEFORE the Workshop with:")
    print("  1. Which lines above say FAIL:")
    for label in failed:
        print(f"       - {label}")
    print("  2. A screenshot of this whole notebook, or the text of any red error.")
    print("  3. Your Python version, printed in Step 2 above:")
    import platform
    print(f"       Python {platform.python_version()}")
    print()
    print("Things worth trying yourself first, in order:")
    print("  a. Runtime -> Restart session, then Runtime -> Run all.")
    print("  b. Open the notebook again in a brand-new Colab session.")
    print("  c. Try a different network (a phone hotspot rules out a firewall).")
    print()
    print("Please do not spend more than ten minutes on this. Send us the")
    print("screenshot and we will sort it out on the day.")
